# Mapeamento dos XPaths e fluxo de teste

Notebook de apoio para validar os seletores usados na automacao.

In [13]:
import base64
from selenium import webdriver
from selenium.common.exceptions import ElementNotInteractableException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [14]:
URL = "https://portaldatransparencia.gov.br/pessoa-fisica/busca/lista?pagina=1&tamanhoPagina=10"
TIMEOUT = 20
TERMO_BUSCA = "A DILA DA SILVA BRITO LIMA"

In [29]:
XPATHS_BUSCA = {
    "input_termo": "//input[@type='search']",
    "bt_refine_busca": "//span[normalize-space()='Refine a Busca']",
    "bt_consultar": "//button[@id='btnConsultarPF']",
    "resultado_primeiro_link": "(//a[contains(@class,'link-busca-nome')])[1]",
}

XPATHS_CHECKBOX = {
    "beneficiario_programa_social": "//label[@for='beneficiarioProgramaSocial']",
    "sancao_vigente": "//label[@for='sancaoVigente']",
    "ocupante_imovel_funcional": "//label[@for='ocupanteImovelFuncional']",
    "possui_contrato": "//label[@for='possuiContrato']",
    "favorecido_recurso_publico": "//label[@for='favorecidoRecurso']",
    "emitente_nfe": "//label[@for='emitenteNfe']",
}

XPATHS_RESULTADO = {
    "nome": "//div[.//strong[normalize-space(.)='Nome']]/span",
    "cpf": "//div[.//strong[normalize-space(.)='CPF']]/span",
    "localidade": "//div[.//strong[normalize-space(.)='Localidade']]/span",
    "recebimentos_recursos": "//*[@id='accordion1']/div[1]/button",
}

XPATHS_BENEFICIOS = {
    "blocos_tabela": "//div[contains(@class,'br-table')][.//div[contains(@class,'responsive')]/strong and .//tbody/tr]",
    "titulo": ".//div[contains(@class,'responsive')]/strong",
    "linhas": ".//tbody/tr",
    "colunas": "./td",
}

In [15]:
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--window-size=1920,1080")
#options.add_argument("--headless=new")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, TIMEOUT)
driver.get(URL)

In [16]:
# Preenche o termo, abre os filtros, marca os checkboxes desejados e consulta
wait.until(EC.element_to_be_clickable((By.XPATH, XPATHS_BUSCA['input_termo']))).send_keys(TERMO_BUSCA)
wait.until(EC.element_to_be_clickable((By.XPATH, XPATHS_BUSCA['bt_refine_busca']))).click()
wait.until(EC.element_to_be_clickable((By.XPATH, XPATHS_CHECKBOX['beneficiario_programa_social']))).click()


In [ ]:
# Clica para fazer a consulta
wait.until(EC.element_to_be_clickable((By.XPATH, XPATHS_BUSCA['bt_consultar']))).click()

In [20]:
# Localiza o nome na listagem, rola ate ele e abre o detalhe da pessoa
resultado = wait.until(EC.visibility_of_element_located((By.XPATH, XPATHS_BUSCA['resultado_primeiro_link'])))
driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", resultado)
resultado.click()

In [30]:
# Abertura da secao de recebimentos de recursos
wait.until(EC.element_to_be_clickable((By.XPATH, XPATHS_RESULTADO['recebimentos_recursos']))).click()

In [ ]:
# Captura dos campos principais
nome = wait.until(EC.visibility_of_element_located((By.XPATH, XPATHS_RESULTADO['nome']))).text
cpf = wait.until(EC.visibility_of_element_located((By.XPATH, XPATHS_RESULTADO['cpf']))).text
localidade = wait.until(EC.visibility_of_element_located((By.XPATH, XPATHS_RESULTADO['localidade']))).text

print(nome)
print(cpf)
print(localidade)

In [ ]:
# Localiza os blocos de tabela de beneficios
tabelas = wait.until(
    lambda d: [
        bloco
        for bloco in d.find_elements(By.XPATH, XPATHS_BENEFICIOS['blocos_tabela'])
        if bloco.is_displayed()
    ]
)

for tabela in tabelas:
    titulo = tabela.find_element(By.XPATH, XPATHS_BENEFICIOS['titulo']).text
    print(titulo)

In [ ]:
# Extracao dos dados das tabelas
dados_beneficios = []

for tabela in tabelas:
    titulo = tabela.find_element(By.XPATH, XPATHS_BENEFICIOS['titulo']).text
    linhas = tabela.find_elements(By.XPATH, XPATHS_BENEFICIOS['linhas'])
    registros = []

    for linha in linhas:
        colunas = linha.find_elements(By.XPATH, XPATHS_BENEFICIOS['colunas'])
        if len(colunas) < 4:
            continue

        link = colunas[0].find_element(By.TAG_NAME, 'a').get_attribute('href')
        registros.append({
            'nis': colunas[1].text.strip(),
            'nome': colunas[2].text.strip(),
            'valor_recebido': colunas[3].text.strip(),
            'detalhar_url': link,
        })

    dados_beneficios.append({
        'programa': titulo,
        'registros': registros,
    })

dados_beneficios

In [ ]:
# Printar as informacoes testadas
prints_tabelas = []

for i, tabela in enumerate(tabelas, start=1):
    titulo = tabela.find_element(By.XPATH, XPATHS_BENEFICIOS['titulo']).text.strip()

    nome_arquivo = f"recebimentos-recursos-tabela-{i}.png"
    tabela.screenshot(nome_arquivo)

    tabela_base64 = base64.b64encode(tabela.screenshot_as_png).decode("utf-8")

    prints_tabelas.append({
        "programa": titulo,
        "arquivo": nome_arquivo,
        "base64": tabela_base64,
    })

print(prints_tabelas)
print(prints_tabelas[0]["base64"][:120])

In [12]:
driver.quit()